In [1]:
!pip install torch tqdm

In [2]:
import os
import math
import zipfile
import urllib.request
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm


In [3]:


# -----------------------------
# 1. Download text8 dataset
# -----------------------------

url = "http://mattmahoney.net/dc/text8.zip"
zip_path = "text8.zip"

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as z:
    text = z.read("text8").decode("utf-8")

print("Dataset loaded")
print("Characters:", len(text))

Dataset loaded
Characters: 100000000


In [4]:



# -----------------------------
# 2. Tokenize text
# -----------------------------

tokens = text.split()

# Use smaller subset for quick training
tokens = tokens[:200000]

print("Total tokens:", len(tokens))


# -----------------------------
# 3. Build vocabulary
# -----------------------------

vocab_size = 5000

word_counts = Counter(tokens)
most_common = word_counts.most_common(vocab_size)

word_to_id = {word: i for i, (word, count) in enumerate(most_common)}
id_to_word = {i: word for word, i in word_to_id.items()}

filtered_tokens = [word_to_id[word] for word in tokens if word in word_to_id]

print("Vocabulary size:", len(word_to_id))


Total tokens: 200000
Vocabulary size: 5000


In [5]:


# -----------------------------
# 4. Build co-occurrence matrix
# -----------------------------

window_size = 5
cooccurrence = defaultdict(float)

for center_pos, center_word_id in tqdm(
    enumerate(filtered_tokens),
    total=len(filtered_tokens),
    desc="Building co-occurrence"
):
    start = max(0, center_pos - window_size)
    end = min(len(filtered_tokens), center_pos + window_size + 1)

    for context_pos in range(start, end):
        if context_pos == center_pos:
            continue

        context_word_id = filtered_tokens[context_pos]
        distance = abs(center_pos - context_pos)

        # Weight nearby words more
        cooccurrence[(center_word_id, context_word_id)] += 1.0 / distance

print("Co-occurrence pairs:", len(cooccurrence))


# -----------------------------
# 5. Prepare training data
# -----------------------------

pairs = list(cooccurrence.keys())
values = list(cooccurrence.values())

i_indices = torch.tensor([p[0] for p in pairs], dtype=torch.long)
j_indices = torch.tensor([p[1] for p in pairs], dtype=torch.long)
x_values = torch.tensor(values, dtype=torch.float32)

print("Training samples:", len(x_values))


Building co-occurrence: 100%|██████████| 178437/178437 [00:03<00:00, 44824.72it/s]


Co-occurrence pairs: 612938
Training samples: 612938


In [6]:


# -----------------------------
# 6. Define GloVe model
# -----------------------------

class GloVeModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)

        self.word_biases = nn.Embedding(vocab_size, 1)
        self.context_biases = nn.Embedding(vocab_size, 1)

        nn.init.xavier_uniform_(self.word_embeddings.weight)
        nn.init.xavier_uniform_(self.context_embeddings.weight)

        nn.init.zeros_(self.word_biases.weight)
        nn.init.zeros_(self.context_biases.weight)

    def forward(self, word_ids, context_ids):
        word_vecs = self.word_embeddings(word_ids)
        context_vecs = self.context_embeddings(context_ids)

        word_bias = self.word_biases(word_ids).squeeze()
        context_bias = self.context_biases(context_ids).squeeze()

        dot_product = torch.sum(word_vecs * context_vecs, dim=1)

        return dot_product + word_bias + context_bias


# -----------------------------
# 7. GloVe loss function
# -----------------------------

def glove_weight(x, x_max=100, alpha=0.75):
    return torch.where(
        x < x_max,
        (x / x_max) ** alpha,
        torch.ones_like(x)
    )

In [7]:
# -----------------------------
# 8. Train model
# -----------------------------

embedding_dim = 50
batch_size = 1024
epochs = 20
learning_rate = 0.05

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = GloVeModel(vocab_size, embedding_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

dataset_size = len(x_values)

for epoch in range(epochs):
    permutation = torch.randperm(dataset_size)

    total_loss = 0

    for start in tqdm(range(0, dataset_size, batch_size), desc=f"Epoch {epoch+1}"):
        indices = permutation[start:start + batch_size]

        batch_i = i_indices[indices].to(device)
        batch_j = j_indices[indices].to(device)
        batch_x = x_values[indices].to(device)

        predictions = model(batch_i, batch_j)

        log_x = torch.log(batch_x)

        weights = glove_weight(batch_x)

        loss = torch.mean(weights * (predictions - log_x) ** 2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")



Using device: cpu


Epoch 1: 100%|██████████| 599/599 [00:03<00:00, 158.28it/s]


Epoch 1, Loss: 65.5956


Epoch 2: 100%|██████████| 599/599 [00:03<00:00, 178.88it/s]


Epoch 2, Loss: 147.9731


...

Epoch 18, Loss: 67.7238


Epoch 19: 100%|██████████| 599/599 [00:03<00:00, 190.32it/s]


Epoch 19, Loss: 115.0412


Epoch 20: 100%|██████████| 599/599 [00:03<00:00, 192.87it/s]

Epoch 20, Loss: 180.6296


In [8]:

# -----------------------------
# 9. Get final word vectors
# -----------------------------

final_embeddings = (
    model.word_embeddings.weight.data +
    model.context_embeddings.weight.data
).cpu()

print("Embedding shape:", final_embeddings.shape)


# -----------------------------
# 10. Find similar words
# -----------------------------

def cosine_similarity(vec1, vec2):
    return torch.dot(vec1, vec2) / (torch.norm(vec1) * torch.norm(vec2))


def most_similar(word, top_n=10):
    if word not in word_to_id:
        print("Word not in vocabulary")
        return

    word_id = word_to_id[word]
    word_vector = final_embeddings[word_id]

    similarities = []

    for i in range(len(id_to_word)):
        if i == word_id:
            continue

        sim = cosine_similarity(word_vector, final_embeddings[i])
        similarities.append((id_to_word[i], sim.item()))

    similarities.sort(key=lambda x: x[1], reverse=True)

    return similarities[:top_n]


print(most_similar("king"))
print(most_similar("computer"))
print(most_similar("music"))

Embedding shape: torch.Size([5000, 50])
[('lycomedes', 0.5057350397109985), ('corresponding', 0.48946109414100647), ('buddhism', 0.47609660029411316), ('cooper', 0.4589653015136719), ('spend', 0.45764583349227905), ('terms', 0.4483761787414551), ('wife', 0.44827190041542053), ('atomic', 0.4450463354587555), ('educated', 0.4450385868549347), ('cognitive', 0.4292643368244171)]
[('entry', 0.48503831028938293), ('situation', 0.477541983127594), ('interests', 0.4660402238368988), ('anconia', 0.45842528343200684), ('interaction', 0.45116329193115234), ('racist', 0.4404308497905731), ('section', 0.42870819568634033), ('cure', 0.42261213064193726), ('transformed', 0.4172680079936981), ('vladimir', 0.4149554371833801)]
[('applied', 0.5087097883224487), ('developmental', 0.5029201507568359), ('disambiguation', 0.4615093171596527), ('watching', 0.45495766401290894), ('propaganda', 0.4305957555770874), ('forum', 0.4273947477340698), ('diacritics', 0.41854628920555115), ('varies', 0.417620360851287

In [9]:
def print_word_vector(word):
    if word not in word_to_id:
        print(f"{word} not found")
        return

    idx = word_to_id[word]
    vec = final_embeddings[idx].numpy()

    print(f"\n{word.upper()} VECTOR")
    print("-" * 50)

    for i, value in enumerate(vec):
        print(f"Dimension {i:02d}: {value:.6f}")


print_word_vector("king")
print_word_vector("computer")
print_word_vector("music")


KING VECTOR
--------------------------------------------------
Dimension 00: 0.598439
Dimension 01: -0.422878
Dimension 02: 1.051683
Dimension 03: 1.656673
Dimension 04: -0.172061
Dimension 05: 1.420222
Dimension 06: 1.951279
Dimension 07: 1.002643
Dimension 08: -0.910590
Dimension 09: -0.037079
Dimension 10: 1.371599
Dimension 11: -0.500909
Dimension 12: 1.006593
Dimension 13: 0.358316
Dimension 14: -1.631873
Dimension 15: 1.088434
Dimension 16: 1.666267
Dimension 17: 1.281697
Dimension 18: 0.956745
Dimension 19: -1.781520
Dimension 20: -1.964288
Dimension 21: -0.813375
Dimension 22: -0.907849
Dimension 23: 3.131246
Dimension 24: -0.732086
Dimension 25: -0.889200
Dimension 26: 0.354798
Dimension 27: -1.395279
Dimension 28: 0.175337
Dimension 29: -1.128932
Dimension 30: 2.087199
Dimension 31: -1.018910
Dimension 32: 0.026934
Dimension 33: -0.107934
Dimension 34: -1.716815
Dimension 35: -0.069654
Dimension 36: -0.106669
Dimension 37: -0.047997
Dimension 38: 0.001061
Dimension 39: -1.95